In [ ]:
%%sql -r dataframe_1
USE DATABASE FINANCE_DB;
USE SCHEMA FINANCE;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import StructType, StructField, StringType

session = get_active_session()
session.use_role('ACCOUNTADMIN')
session.use_database('FINANCE_DB')
session.use_schema('FINANCE')

In [ ]:
%%sql -r hsbc_files
LIST @FINANCE_DB.FINANCE.finance_sources_sse/HSBC/

In [ ]:
%%sql -r raw_extract
CREATE OR REPLACE TABLE FINANCE_DB.FINANCE.HSBC_RAW_EXTRACT AS
WITH pdf_files AS (
    SELECT relative_path
    FROM DIRECTORY(@FINANCE_DB.FINANCE.finance_sources_sse)
    WHERE relative_path LIKE 'HSBC/2026/%.pdf'
),
parsed AS (
    SELECT
        relative_path,
        AI_PARSE_DOCUMENT(
            TO_FILE('@FINANCE_DB.FINANCE.finance_sources_sse', relative_path),
            {'mode': 'LAYOUT'}
        ):content::VARCHAR AS content
    FROM pdf_files
),
extracted AS (
    SELECT
        relative_path,
        AI_COMPLETE(
            'claude-4-sonnet',
            'Extract ALL bank transactions from the following HSBC bank statement into a JSON array.'
            || 'Each transaction object must have these exact fields: '
            || '"date" (DD/MM/YYYY format, fill in year from statement period), '
            || '"payment_type" (DD, VIS, SO, BP, CR, TFR, ATM, etc.), '
            || '"description" (full text of payee/details), '
            || '"paid_out" (number or null), '
            || '"paid_in" (number or null), '
            || '"balance" (number or null). '
            || 'Skip BALANCE BROUGHT FORWARD and BALANCE CARRIED FORWARD rows. '
            || 'Return ONLY a valid JSON array, no other text.\n\n'
            || content,
            {'max_tokens': 28192}
        ) AS transactions_json
    FROM parsed
)
SELECT
    relative_path,
    transactions_json
FROM extracted;

In [ ]:
%%sql -r hsbc_transactions
--CREATE OR REPLACE TABLE FINANCE_DB.FINANCE.HSBC_TRANSACTIONS AS
INSERT INTO FINANCE_DB.FINANCE.HSBC_TRANSACTIONS
WITH cleaned AS (
    SELECT
        relative_path,
        REGEXP_REPLACE(
            REGEXP_REPLACE(transactions_json, '^"?```json\\s*', ''),
            '\\s*```"?$', ''
        ) AS clean_json
    FROM FINANCE_DB.FINANCE.HSBC_RAW_EXTRACT
),
trimmed AS (
    SELECT
        relative_path,
        CASE
            WHEN clean_json LIKE '"[%' THEN SUBSTR(clean_json, 2, LENGTH(clean_json) - 2)
            ELSE clean_json
        END AS final_json
    FROM cleaned
)
SELECT
    SPLIT_PART(r.relative_path, '/', -1) AS SOURCE_FILE,
    t.value:date::VARCHAR AS DATE,
    t.value:payment_type::VARCHAR AS PAYMENT_TYPE,
    t.value:description::VARCHAR AS DESCRIPTION,
    t.value:paid_out::NUMBER(10,2) AS PAID_OUT,
    t.value:paid_in::NUMBER(10,2) AS PAID_IN,
    t.value:balance::NUMBER(10,2) AS BALANCE
FROM trimmed r,
    LATERAL FLATTEN(input => TRY_PARSE_JSON(REPLACE(r.final_json, '\\n', ''))) t
WHERE NOT EXISTS (SELECT 1 FROM FINANCE_DB.FINANCE.HSBC_TRANSACTIONS t WHERE t.SOURCE_FILE = SPLIT_PART(r.relative_path, '/', -1));

In [ ]:
%%sql -r preview
CREATE OR REPLACE VIEW FINANCE_DB.FINANCE.HSBC_TRANSACTIONS_VW
AS
SELECT
    TRY_TO_DATE(DATE, 'DD/MM/YYYY') AS TRANSACTION_DATE,
    PAYMENT_TYPE,
    DESCRIPTION,
    PAID_OUT,
    PAID_IN,
    BALANCE,
    SOURCE_FILE
FROM FINANCE_DB.FINANCE.HSBC_TRANSACTIONS;

In [ ]:
%%sql -r dataframe_7
--SELECT count(distinct DATE_TRUNC('month', TRANSACTION_DATE)) FROM FINANCE_DB.FINANCE.HSBC_TRANSACTIONS_VW;
--ORDER BY  TRANSACTION_DATE LIMIT 100;
-- select  TRY_TO_DATE(DATE, 'DD/MM/YYYY') as TRANSACTION_DATE, * from FINANCE_DB.FINANCE.HSBC_TRANSACTIONS order by  TRY_TO_DATE(DATE, 'DD/MM/YYYY');
select RELATIVE_PATH from FINANCE_DB.FINANCE.HSBC_RAW_EXTRACT order by 1;

In [ ]:
%%sql -r monthly_summary
SELECT
    DATE_TRUNC('month', TRANSACTION_DATE) AS MONTH,
    COUNT(1) AS TRAN_CNT,
    SUM(PAID_OUT) AS TOTAL_PAID_OUT,
    SUM(PAID_IN) AS TOTAL_PAID_IN
FROM FINANCE_DB.FINANCE.HSBC_TRANSACTIONS_VW
GROUP BY MONTH
ORDER BY MONTH